# Llibreries

In [1]:
import pandas as pd
import numpy as np
import json
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [2]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")

In [3]:
dim_barris.head()

,codi_districte,nom_districte,codi_barri,nom_barri,geometria_etrs89,geometria_wgs84
0,1,Ciutat Vella,1,el Raval,"POLYGON ((430162.1875 4581936.9845, 430102.838...","POLYGON ((2.16471378585589 41.3859301967194, 2..."
1,1,Ciutat Vella,2,el Barri Gòtic,"POLYGON ((431189.9075 4581851.4475, 431025.789...","POLYGON ((2.1770141884741 41.385248355328, 2.1..."
2,1,Ciutat Vella,3,la Barceloneta,"POLYGON ((432798.7341255 4582081.2599495, 4327...","POLYGON ((2.19622882469513 41.387454220446, 2...."
3,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera","POLYGON ((431733.736 4582441.816, 431557.5115 ...","POLYGON ((2.18345134701381 41.3906119681235, 2..."
4,2,Eixample,5,el Fort Pienc,"POLYGON ((431741.8152 4582625.6491, 432012.183...","POLYGON ((2.18352725722411 41.3922683849226, 2..."


In [4]:
dimensions.head()

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN
0,1,SEXE,1,Dona,Mujer,Female
1,1,SEXE,2,Home,Hombre,Male
2,2,EDAT_1,0,0 anys,0 años,0 years
3,2,EDAT_1,1,1 anys,1 años,1 years
4,2,EDAT_1,2,2 anys,2 años,2 years


# Dades demogràfiques
Aquests datasets seran utilitzats per extreure atributs demogràfic per al conjunt agregat final.
- Població per barri (Recompte)
- Població per continent de nacionalitat (Recompte)
- Estudis Poblacionals per edat i lloc de naixement

## Població Total per Barri
Amb aquests arxius obtindrem la població total per barri.

In [5]:
# Import dades
df_poblacio_total_23_raw = pd.read_csv("../data/raw/poblacio/2023_pad_mdbas.csv")
df_poblacio_total_23_raw.head()


,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,AEB,Seccio_Censal,Valor
0,2023-01-01,1,Ciutat Vella,1,el Raval,1,1001,1246
1,2023-01-01,1,Ciutat Vella,1,el Raval,1,1002,1238
2,2023-01-01,1,Ciutat Vella,1,el Raval,2,1003,3406
3,2023-01-01,1,Ciutat Vella,1,el Raval,2,1004,2881
4,2023-01-01,1,Ciutat Vella,1,el Raval,3,1005,2294


In [6]:
df_poblacio_total_15_raw = pd.read_csv("../data/raw/poblacio/2015_pad_mdbas.csv")
df_poblacio_total_15_raw.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,AEB,Seccio_Censal,Valor
0,2015-01-01,1,Ciutat Vella,1,el Raval,1,1001,1321
1,2015-01-01,1,Ciutat Vella,1,el Raval,1,1002,1528
2,2015-01-01,1,Ciutat Vella,1,el Raval,2,1003,3340
3,2015-01-01,1,Ciutat Vella,1,el Raval,2,1004,2785
4,2015-01-01,1,Ciutat Vella,1,el Raval,3,1005,2460


In [7]:
df_poblacio_total_23_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  1068 non-null   object
 1   Codi_Districte   1068 non-null   int64 
 2   Nom_Districte    1068 non-null   object
 3   Codi_Barri       1068 non-null   int64 
 4   Nom_Barri        1068 non-null   object
 5   AEB              1068 non-null   int64 
 6   Seccio_Censal    1068 non-null   int64 
 7   Valor            1068 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 66.9+ KB


**Observacions:**
- No hi ha presència de valors nuls
- Atribut valor en format correcte

In [8]:
df_poblacio_total_15_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  1068 non-null   object
 1   Codi_Districte   1068 non-null   int64 
 2   Nom_Districte    1068 non-null   object
 3   Codi_Barri       1068 non-null   int64 
 4   Nom_Barri        1068 non-null   object
 5   AEB              1068 non-null   int64 
 6   Seccio_Censal    1068 non-null   int64 
 7   Valor            1068 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 66.9+ KB


**Observacions:**
- Igual que amb les dades de 2023

In [9]:
df_poblacio_total_15 = df_poblacio_total_15_raw.groupby("Codi_Barri").agg({"Valor": "sum"}).reset_index()
df_poblacio_total_23 = df_poblacio_total_23_raw.groupby("Codi_Barri").agg({"Valor": "sum"}).reset_index()

In [10]:
df_poblacio_total_23.rename(columns={"Valor": "Poblacio_Total"}, inplace=True)
df_poblacio_total_15.rename(columns={"Valor": "Poblacio_Total"}, inplace=True)

## Població per barri per Nacionalitat (Esp, resta Unió Europea, resta món)
Per aquest indicador es processaran dos fitxers per als dos anys. En un primer, obtindrem la població espanyola resident als barris. Posteriorment, utilitzarem un segon conjunt de dades amb la població desglosada per continents de nacionalitat. A partir d'aquests dos fitxers podrem extreure indicadors relacionats amb les proporcions de població estrangera en els barris, i inclos segmentar-la per continents. 

In [11]:
# Població

df_poblacio_23 = pd.read_csv('../data/raw/poblacio/2023_pad_mdbas_nacionalitat-g_sexe.csv')
df_poblacio_23.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,AEB,Seccio_Censal,Valor,NACIONALITAT_G,SEXE
0,2023-01-01,1,Ciutat Vella,1,el Raval,1,1001,315,1,1
1,2023-01-01,1,Ciutat Vella,1,el Raval,1,1001,357,1,2
2,2023-01-01,1,Ciutat Vella,1,el Raval,1,1001,68,2,1
3,2023-01-01,1,Ciutat Vella,1,el Raval,1,1001,65,2,2
4,2023-01-01,1,Ciutat Vella,1,el Raval,1,1001,223,3,1


In [12]:
df_poblacio_23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6646 entries, 0 to 6645
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  6646 non-null   object
 1   Codi_Districte   6646 non-null   int64 
 2   Nom_Districte    6646 non-null   object
 3   Codi_Barri       6646 non-null   int64 
 4   Nom_Barri        6646 non-null   object
 5   AEB              6646 non-null   int64 
 6   Seccio_Censal    6646 non-null   int64 
 7   Valor            6646 non-null   object
 8   NACIONALITAT_G   6646 non-null   int64 
 9   SEXE             6646 non-null   int64 
dtypes: int64(6), object(4)
memory usage: 519.3+ KB


**Observacions:**
- No es detecten nuls
- Atribut de valor identificat com a object.

In [13]:
# Població

df_poblacio_15 = pd.read_csv('../data/raw/poblacio/2015_pad_mdbas_nacionalitat-g_sexe.csv')
df_poblacio_15.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,AEB,Seccio_Censal,Valor,NACIONALITAT_G,SEXE
0,2015-01-01,1,Ciutat Vella,1,el Raval,1,1001,345,1,1
1,2015-01-01,1,Ciutat Vella,1,el Raval,1,1001,397,1,2
2,2015-01-01,1,Ciutat Vella,1,el Raval,1,1001,68,2,1
3,2015-01-01,1,Ciutat Vella,1,el Raval,1,1001,79,2,2
4,2015-01-01,1,Ciutat Vella,1,el Raval,1,1001,160,3,1


In [14]:
df_poblacio_15.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6582 entries, 0 to 6581
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  6582 non-null   object
 1   Codi_Districte   6582 non-null   int64 
 2   Nom_Districte    6582 non-null   object
 3   Codi_Barri       6582 non-null   int64 
 4   Nom_Barri        6582 non-null   object
 5   AEB              6582 non-null   int64 
 6   Seccio_Censal    6582 non-null   int64 
 7   Valor            6582 non-null   object
 8   NACIONALITAT_G   6582 non-null   int64 
 9   SEXE             6582 non-null   int64 
dtypes: int64(6), object(4)
memory usage: 514.3+ KB


**Observacions:**
- Mateixes observacions que al dataset de 2023

In [15]:
df_poblacio_23[df_poblacio_23["Valor"] == ".."]

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,AEB,Seccio_Censal,Valor,NACIONALITAT_G,SEXE
18,2023-01-01,1,Ciutat Vella,1,el Raval,2,1003,..,4,1
19,2023-01-01,1,Ciutat Vella,1,el Raval,2,1003,..,4,2
26,2023-01-01,1,Ciutat Vella,1,el Raval,2,1004,..,4,1
33,2023-01-01,1,Ciutat Vella,1,el Raval,3,1005,..,4,1
52,2023-01-01,1,Ciutat Vella,1,el Raval,4,1008,..,4,1
...,...,...,...,...,...,...,...,...,...,...
6517,2023-01-01,10,Sant Martí,72,Sant Martí de Provençals,229,10122,..,4,1
6538,2023-01-01,10,Sant Martí,73,la Verneda i la Pau,230,10126,..,2,1
6539,2023-01-01,10,Sant Martí,73,la Verneda i la Pau,230,10126,..,2,2
6578,2023-01-01,10,Sant Martí,73,la Verneda i la Pau,231,10132,..,4,2


**Observacions**
- Segons el portal de dades, els valors marcats com a ".." són menors de 5.
- Per tal de no eliminar del dataset aquests valors, els imputem amb un número dins del rang per tal de mantenir informació sense esbiaixar els resultats.

In [16]:
# Convertim valors ".." a NaN 
df_poblacio_23["Valor"] = df_poblacio_23["Valor"].replace("..", 4)
df_poblacio_15["Valor"] = df_poblacio_15["Valor"].replace("..", 4)

In [17]:
# Convertir la columna "Valor" a int32
df_poblacio_23["Valor"] = df_poblacio_23["Valor"].astype(str).astype("Int32")
df_poblacio_15["Valor"] = df_poblacio_15["Valor"].astype(str).astype("Int32")

In [18]:
# Agregar a població per barri i nacionalitat. Ignorem sexe
df_poblacio_15_agg = df_poblacio_15.groupby(["Codi_Barri", "NACIONALITAT_G"]).agg({"Valor": "sum"}).reset_index()
df_poblacio_23_agg = df_poblacio_23.groupby(["Codi_Barri", "NACIONALITAT_G"]).agg({"Valor": "sum"}).reset_index()

In [19]:
df_poblacio_15_agg.head()

,Codi_Barri,NACIONALITAT_G,Valor
0,1,1,24498
1,1,2,4347
2,1,3,18293
3,1,4,44
4,2,1,9007


In [20]:
df_poblacio_23_agg.head()

,Codi_Barri,NACIONALITAT_G,Valor
0,1,1,22396
1,1,2,4563
2,1,3,18693
3,1,4,56
4,2,1,8385


Creuarem amb la dimensió de nacionalitats i posteriorment pivotarem la columna per obtenir una columna per cada tipus de nacionalitat.

In [21]:
# Definim la dimensió
dim_nacionalitat_g = dimensions[dimensions["Desc_Dimensio"] == "NACIONALITAT_G"]
dim_nacionalitat_g.head()

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN
176,7,NACIONALITAT_G,1,Espanya,España,Spain
177,7,NACIONALITAT_G,2,Resta de la Unió Europea,Resto de la Unión Europea,Rest of European Union
178,7,NACIONALITAT_G,3,Resta del món,Resto del mundo,Rest of World
179,7,NACIONALITAT_G,4,No consta,No consta,Not available


In [22]:
df_poblacio_23_merged = df_poblacio_23_agg.merge(dim_nacionalitat_g[["Codi_Valor","Desc_Valor_CA"]], left_on="NACIONALITAT_G", right_on="Codi_Valor", how="left")
df_poblacio_23_merged.head()

,Codi_Barri,NACIONALITAT_G,Valor,Codi_Valor,Desc_Valor_CA
0,1,1,22396,1,Espanya
1,1,2,4563,2,Resta de la Unió Europea
2,1,3,18693,3,Resta del món
3,1,4,56,4,No consta
4,2,1,8385,1,Espanya


In [23]:
df_poblacio_15_merged = df_poblacio_15_agg.merge(dim_nacionalitat_g[["Codi_Valor","Desc_Valor_CA"]], left_on="NACIONALITAT_G", right_on="Codi_Valor", how="left")
df_poblacio_15_merged.head()

,Codi_Barri,NACIONALITAT_G,Valor,Codi_Valor,Desc_Valor_CA
0,1,1,24498,1,Espanya
1,1,2,4347,2,Resta de la Unió Europea
2,1,3,18293,3,Resta del món
3,1,4,44,4,No consta
4,2,1,9007,1,Espanya


In [24]:
# Pivotem
df_pob_23_pivot = df_poblacio_23_merged.pivot_table(index=["Codi_Barri"], columns="Desc_Valor_CA", values=["Valor"], aggfunc= "sum", fill_value=0).droplevel(0, axis=1).reset_index()
df_pob_15_pivot = df_poblacio_15_merged.pivot_table(index=["Codi_Barri"], columns="Desc_Valor_CA", values=["Valor"], aggfunc= "sum",fill_value=0).droplevel(0, axis=1).reset_index()

In [25]:
# Netejem indexs i columnes
df_pob_23_pivot.columns.name = None
df_pob_23_pivot = df_pob_23_pivot.reset_index(drop=True)

df_pob_15_pivot.columns.name = None
df_pob_15_pivot = df_pob_15_pivot.reset_index(drop=True)

## Població per Regió (Amèrca del Nord, Europa Occidental, etc.)
Amb aquestes dades podrem desglosar la població segons la regió, i d'aquesta manera, extreure dades més detallades sobre la població estrangera i quin orígen tenen. 

In [26]:
df_nacionalitats_23 = pd.read_csv("../data/raw/poblacio/2023_pad_mdb_nacionalitat-regio_sexe.csv")
df_nacionalitats_15 = pd.read_csv("../data/raw/poblacio/2015_pad_mdb_nacionalitat-regio_sexe.csv")

In [27]:
dim_regions = dimensions[dimensions["Desc_Dimensio"] == "NACIONALITAT_REGIO"]

In [28]:
df_nacionalitats_23.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,NACIONALITAT_REGIO,SEXE
0,2023-01-01,1,Ciutat Vella,1,el Raval,23,1,1
1,2023-01-01,1,Ciutat Vella,1,el Raval,9,1,2
2,2023-01-01,1,Ciutat Vella,1,el Raval,13,2,1
3,2023-01-01,1,Ciutat Vella,1,el Raval,17,2,2
4,2023-01-01,1,Ciutat Vella,1,el Raval,604,3,1


In [29]:
df_nacionalitats_23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2625 entries, 0 to 2624
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Data_Referencia     2625 non-null   object
 1   Codi_Districte      2625 non-null   int64 
 2   Nom_Districte       2625 non-null   object
 3   Codi_Barri          2625 non-null   int64 
 4   Nom_Barri           2625 non-null   object
 5   Valor               2625 non-null   object
 6   NACIONALITAT_REGIO  2625 non-null   int64 
 7   SEXE                2625 non-null   int64 
dtypes: int64(4), object(4)
memory usage: 164.2+ KB


**Observacions:**
- No hi ha nuls, però Valor torna a estar com a obj, indicant que hi pot haver valors ".."

In [30]:
df_nacionalitats_15.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,NACIONALITAT_REGIO,SEXE
0,2015-01-01,1,Ciutat Vella,1,el Raval,..,1,1
1,2015-01-01,1,Ciutat Vella,1,el Raval,..,1,2
2,2015-01-01,1,Ciutat Vella,1,el Raval,5,2,1
3,2015-01-01,1,Ciutat Vella,1,el Raval,14,2,2
4,2015-01-01,1,Ciutat Vella,1,el Raval,621,3,1


In [31]:
# Convertim valors ".." a NaN 
df_nacionalitats_15["Valor"] = df_nacionalitats_15["Valor"].replace("..", 4)
df_nacionalitats_23["Valor"] = df_nacionalitats_23["Valor"].replace("..", 4)

In [32]:
# Convertir la columna "Valor" a int32
df_nacionalitats_23["Valor"] = df_nacionalitats_23["Valor"].astype(str).astype("Int32")
df_nacionalitats_15["Valor"] = df_nacionalitats_15["Valor"].astype(str).astype("Int32")

In [33]:
# Agregar a població per barri i nacionalitat. Ignorem sexe
df_nacionalitats_23_agg = df_nacionalitats_23.groupby(["Codi_Barri", "NACIONALITAT_REGIO"]).agg({"Valor": "sum"}).reset_index()
df_nacionalitats_15_agg = df_nacionalitats_15.groupby(["Codi_Barri", "NACIONALITAT_REGIO"]).agg({"Valor": "sum"}).reset_index()

In [34]:
df_nacionalitats_23_merged =df_nacionalitats_23_agg.merge(dim_regions[["Codi_Valor","Desc_Valor_CA"]], left_on="NACIONALITAT_REGIO", right_on="Codi_Valor", how="left")
df_nacionalitats_15_merged =df_nacionalitats_15_agg.merge(dim_regions[["Codi_Valor","Desc_Valor_CA"]], left_on="NACIONALITAT_REGIO", right_on="Codi_Valor", how="left")

In [35]:
df_nacionalitats_23_pivot = df_nacionalitats_23_merged.pivot_table(index=["Codi_Barri"], columns="Desc_Valor_CA", values=["Valor"], aggfunc= "sum",fill_value=0).droplevel(0, axis=1).reset_index()
df_nacionalitats_15_pivot = df_nacionalitats_15_merged.pivot_table(index=["Codi_Barri"], columns="Desc_Valor_CA", values=["Valor"], aggfunc= "sum",fill_value=0).droplevel(0, axis=1).reset_index()

In [36]:
df_nacionalitats_15_pivot.columns.name = None
df_nacionalitats_15_pivot = df_nacionalitats_15_pivot.reset_index(drop=True)

df_nacionalitats_23_pivot.columns.name = None
df_nacionalitats_23_pivot = df_nacionalitats_23_pivot.reset_index(drop=True)

## Estudis per Població (Esp, resta Unió Europea, Resta Món)
Amb aquestes dades extraurem un indicador percentual per saber quina porció de població té estudis universitaris segons origen, amb focus entre població espanyola i europeus i resta del món. És una variable que alguns autors com Cocola Gant (2020) indiquen que pot ser un factor important per a la gentrificació.

In [40]:
df_estudis_raw_23 = pd.read_csv("../data/raw/educacio/2023_pad_mdb_niv-educa-esta_edat-lloc-naix.csv")
df_estudis_raw_23.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,NIV_EDUCA_esta,LLOC_NAIX
0,2023-01-01,1,Ciutat Vella,1,el Raval,109,1,1
1,2023-01-01,1,Ciutat Vella,1,el Raval,22,1,2
2,2023-01-01,1,Ciutat Vella,1,el Raval,204,1,3
3,2023-01-01,1,Ciutat Vella,1,el Raval,..,1,4
4,2023-01-01,1,Ciutat Vella,1,el Raval,70,1,5


In [41]:
df_estudis_raw_23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2281 entries, 0 to 2280
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  2281 non-null   object
 1   Codi_Districte   2281 non-null   int64 
 2   Nom_Districte    2281 non-null   object
 3   Codi_Barri       2281 non-null   int64 
 4   Nom_Barri        2281 non-null   object
 5   Valor            2281 non-null   object
 6   NIV_EDUCA_esta   2281 non-null   int64 
 7   LLOC_NAIX        2281 non-null   int64 
dtypes: int64(4), object(4)
memory usage: 142.7+ KB


**Observacions**
- No hi ha presència de valors nuls. 
- Atribut valor com a Object, indicant que hi ha valors menors de 5 i que s' hauran d'imputar. 

In [43]:
df_estudis_raw_15 = pd.read_csv("../data/raw/educacio/2015_pad_mdb_niv-educa-esta_edat-lloc-naix.csv")
df_estudis_raw_15.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,NIV_EDUCA_esta,LLOC_NAIX
0,2015-01-01,1,Ciutat Vella,1,el Raval,575,1,1
1,2015-01-01,1,Ciutat Vella,1,el Raval,103,1,2
2,2015-01-01,1,Ciutat Vella,1,el Raval,990,1,3
3,2015-01-01,1,Ciutat Vella,1,el Raval,25,1,4
4,2015-01-01,1,Ciutat Vella,1,el Raval,849,1,5


In [44]:
df_estudis_raw_15.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2276 entries, 0 to 2275
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  2276 non-null   object
 1   Codi_Districte   2276 non-null   int64 
 2   Nom_Districte    2276 non-null   object
 3   Codi_Barri       2276 non-null   int64 
 4   Nom_Barri        2276 non-null   object
 5   Valor            2276 non-null   object
 6   NIV_EDUCA_esta   2276 non-null   int64 
 7   LLOC_NAIX        2276 non-null   int64 
dtypes: int64(4), object(4)
memory usage: 142.4+ KB


**Observacions**
- No hi ha presència de valors nuls. 
- Atribut valor com a Object, indicant que hi ha valors menors de 5 i que s' hauran d'imputar. 

In [45]:
# Mostrem aquests valors
df_estudis_raw_15[df_estudis_raw_15["Valor"] == ".."][:5]

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,NIV_EDUCA_esta,LLOC_NAIX
16,2015-01-01,1,Ciutat Vella,1,el Raval,..,3,6
22,2015-01-01,1,Ciutat Vella,1,el Raval,..,4,6
54,2015-01-01,1,Ciutat Vella,2,el Barri Gòtic,..,4,6
60,2015-01-01,1,Ciutat Vella,2,el Barri Gòtic,..,5,6
62,2015-01-01,1,Ciutat Vella,2,el Barri Gòtic,..,6,2


In [46]:
# Convertim valors ".." a NaN 
df_estudis_raw_23["Valor"] = df_estudis_raw_23["Valor"].replace("..", 4)
df_estudis_raw_15["Valor"] = df_estudis_raw_15["Valor"].replace("..", 4)

In [47]:
# Convertir la columna "Valor" a int32
df_estudis_raw_23["Valor"] = df_estudis_raw_23["Valor"].astype(str).astype("Int32")
df_estudis_raw_15["Valor"] = df_estudis_raw_15["Valor"].astype(str).astype("Int32")

In [48]:
# Agreguem els datasets a nivell de barri, nacionalitat i nivell educatiu
df_estudis_23_agg = df_estudis_raw_23.groupby(["Codi_Barri", "LLOC_NAIX",  "NIV_EDUCA_esta"]).agg({"Valor": "sum"}).reset_index()
df_estudis_15_agg = df_estudis_raw_15.groupby(["Codi_Barri", "LLOC_NAIX", "NIV_EDUCA_esta"]).agg({"Valor": "sum"}).reset_index()

In [49]:
# Creem les dimensions per obtenir les descripcions
dim_estudis = dimensions[dimensions["Desc_Dimensio"] == "NIV_EDUCA_esta"]
dim_lloc_naix = dimensions[dimensions["Desc_Dimensio"] == "LLOC_NAIX"]

Per a les dades del lloc_de_naix, no ens interessa per ara tenir Espanya de manera desglosada. Per tant, crearem una columna customitzada per tal de poder agrupar a la població espanyola.

In [50]:
print("Possibles valors:\n", dim_lloc_naix["Desc_Valor_CA"].unique())

Possibles valors:
 ['Barcelona ciutat' 'Resta de Catalunya' 'Resta d’Espanya'
 'Resta de la Unió Europea' 'Resta del món' 'No consta']


In [51]:
dim_lloc_naix.loc[:,["custom_mapping_regio"]] = dim_lloc_naix["Desc_Valor_CA"].map({
    "Barcelona ciutat": "Espanya",
    "Resta de Catalunya": "Espanya",
    "Resta d’Espanya": "Espanya",
    "Resta de la Unió Europea": "Resta de la Unió Europea",
    "Resta del món": "Resta del món",
    "No consta": "No Consta"
})

dim_lloc_naix.head()

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN,custom_mapping_regio
126,4,LLOC_NAIX,1,Barcelona ciutat,Barcelona ciudad,City of Barcelona,Espanya
127,4,LLOC_NAIX,2,Resta de Catalunya,Resto de Cataluña,Rest of Catalonia,Espanya
128,4,LLOC_NAIX,3,Resta d’Espanya,Resto de España,Rest of Spain,Espanya
129,4,LLOC_NAIX,4,Resta de la Unió Europea,Resto de la Unión Europea,Rest of European Union,Resta de la Unió Europea
130,4,LLOC_NAIX,5,Resta del món,Resto del mundo,Rest of World,Resta del món


In [52]:
# Creuem amb les dimensions per obtenir les descripcions
df_estudis_23_merged = df_estudis_23_agg.merge(dim_estudis[["Codi_Valor","Desc_Valor_CA"]], left_on="NIV_EDUCA_esta", right_on="Codi_Valor", how="left", suffixes=('', '_educ'))\
                                        .merge(dim_lloc_naix[["Codi_Valor","custom_mapping_regio"]], left_on="LLOC_NAIX", right_on="Codi_Valor", how="left", suffixes=('', '_lloc'))

df_estudis_23_merged.head()

,Codi_Barri,LLOC_NAIX,NIV_EDUCA_esta,Valor,Codi_Valor,Desc_Valor_CA,Codi_Valor_lloc,custom_mapping_regio
0,1,1,1,109,1,Sense estudis,1,Espanya
1,1,1,2,1350,2,"Estudis primaris, certificat d'escolaritat, EGB",1,Espanya
2,1,1,3,2356,3,"Batxillerat elemental, graduat escolar, ESO, FPI",1,Espanya
3,1,1,4,1634,4,"Batxillerat superior, BUP, COU, FPII, CFGM gra...",1,Espanya
4,1,1,5,1328,5,"Estudis universitaris, CFGS grau superior",1,Espanya


In [53]:
# Creuem amb les dimensions per obtenir les descripcions
df_estudis_15_merged = df_estudis_15_agg.merge(dim_estudis[["Codi_Valor","Desc_Valor_CA"]], left_on="NIV_EDUCA_esta", right_on="Codi_Valor", how="left", suffixes=('', '_educ'))\
                                        .merge(dim_lloc_naix[["Codi_Valor","custom_mapping_regio"]], left_on="LLOC_NAIX", right_on="Codi_Valor", how="left", suffixes=('', '_lloc'))

df_estudis_15_merged.head()

,Codi_Barri,LLOC_NAIX,NIV_EDUCA_esta,Valor,Codi_Valor,Desc_Valor_CA,Codi_Valor_lloc,custom_mapping_regio
0,1,1,1,575,1,Sense estudis,1,Espanya
1,1,1,2,1723,2,"Estudis primaris, certificat d'escolaritat, EGB",1,Espanya
2,1,1,3,2703,3,"Batxillerat elemental, graduat escolar, ESO, FPI",1,Espanya
3,1,1,4,1905,4,"Batxillerat superior, BUP, COU, FPII, CFGM gra...",1,Espanya
4,1,1,5,1432,5,"Estudis universitaris, CFGS grau superior",1,Espanya


In [54]:
df_estudis_23_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2281 entries, 0 to 2280
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Codi_Barri            2281 non-null   int64 
 1   LLOC_NAIX             2281 non-null   int64 
 2   NIV_EDUCA_esta        2281 non-null   int64 
 3   Valor                 2281 non-null   Int32 
 4   Codi_Valor            2281 non-null   int64 
 5   Desc_Valor_CA         2281 non-null   object
 6   Codi_Valor_lloc       2281 non-null   int64 
 7   custom_mapping_regio  2281 non-null   object
dtypes: Int32(1), int64(5), object(2)
memory usage: 136.0+ KB


En aquest cas, només volem en scope estudis universitaris i gent sense estudis o amb nivell màxim de primària. Per tant, filtrarem previament els datasets per després pivotar.

In [55]:
dim_estudis

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN
203,9,NIV_EDUCA_esta,1,Sense estudis,Sin estudios,Less than primary education
204,9,NIV_EDUCA_esta,2,"Estudis primaris, certificat d'escolaritat, EGB","Estudios primarios, certificado de escolaridad...",Primary education
205,9,NIV_EDUCA_esta,3,"Batxillerat elemental, graduat escolar, ESO, FPI","Bachillerato elemental, graduado escolar, ESO,...",Lower secondary education
206,9,NIV_EDUCA_esta,4,"Batxillerat superior, BUP, COU, FPII, CFGM gra...","Bachillerato superior, BUP, COU, FPII, CFGM gr...",Upper secondary or post-secondary non-tertiary...
207,9,NIV_EDUCA_esta,5,"Estudis universitaris, CFGS grau superior","Estudios universitarios, CFGS grado superior",Tertiary education
208,9,NIV_EDUCA_esta,6,No consta,No consta,Not available


Exclourem els codis 3, 4 i 6

In [56]:
df_estudis_23_filt = df_estudis_23_merged[~df_estudis_23_merged["Codi_Valor"].isin([3,4,6])]
df_estudis_15_filt = df_estudis_15_merged[~df_estudis_15_merged["Codi_Valor"].isin([3,4,6])]

In [57]:
df_estudis_23_pivot = df_estudis_23_filt.pivot_table(index="Codi_Barri", columns=["Desc_Valor_CA", "custom_mapping_regio"], values="Valor", aggfunc = "sum",  fill_value=0).reset_index()
df_estudis_15_pivot = df_estudis_15_filt.pivot_table(index="Codi_Barri", columns=["Desc_Valor_CA", "custom_mapping_regio"], values="Valor", aggfunc = "sum", fill_value=0).reset_index()

In [58]:
df_estudis_15_pivot.head()

Desc_Valor_CA        Codi_Barri  \
custom_mapping_regio              
0                             1   
1                             2   
2                             3   
3                             4   
4                             5   

Desc_Valor_CA        Estudis primaris, certificat d'escolaritat, EGB  \
custom_mapping_regio                                         Espanya   
0                                                               3754   
1                                                               1176   
2                                                               2074   
3                                                               1906   
4                                                               2575   

Desc_Valor_CA                                                          \
custom_mapping_regio No Consta Resta de la Unió Europea Resta del món   
0                            5                      286          8664   
1                            0                      140          1040   
2                            0                       78           751   
3                            4                       89          1497   
4                            4                      103           996   

Desc_Valor_CA        Estudis universitaris, CFGS grau superior            \
custom_mapping_regio                                   Espanya No Consta   
0                                                         3192         6   
1                                                         1962         4   
2                                                          976         4   
3                                                         2717         9   
4                                                         6784         6   

Desc_Valor_CA                                               Sense estudis  \
custom_mapping_regio Resta de la Unió Europea Resta del món       Espanya   
0                                        1977          3552          1668   
1                                        1485          1728           519   
2                                        1080          1173          1086   
3                                        2172          2377           845   
4                                        1060          2061           832   

Desc_Valor_CA                                                          
custom_mapping_regio No Consta Resta de la Unió Europea Resta del món  
0                            0                       25           849  
1                            0                       12           114  
2                            0                       15            80  
3                            0                       14           212  
4                            0                        8            66

In [59]:
# Neteja de columnes per claredat
df_estudis_23_pivot.columns = [
    nivell if nivell == "Codi_Barri" else f"{nivell.replace("\xa0", " ").strip()}_edu_{regio}" for nivell, regio in df_estudis_23_pivot.columns
]


df_estudis_15_pivot.columns = [
    nivell if nivell == "Codi_Barri" else f"{nivell.replace("\xa0", " ").strip()}_edu_{regio}" for nivell, regio in df_estudis_15_pivot.columns
]

In [60]:
df_estudis_23_pivot.columns.name = None
df_estudis_15_pivot.columns.name = None

In [61]:
# Netejem per preparar per als joins posteriors
# Canvi de noms
df_estudis_23_pivot = df_estudis_23_pivot.rename(columns = 
                                                 {"Estudis universitaris, CFGS grau superior_edu_Espanya": "estudis_superiors_espanya", 
                                                  "Estudis universitaris, CFGS grau superior_edu_Resta de la Unió Europea": "estudis_superiors_ue", 
                                                  "Estudis universitaris, CFGS grau superior_edu_Resta del món": "estudis_superiors_mon"
                                                  })

# Càlcul de columnes
df_estudis_23_pivot["estudis_inferiors_espanya"] = df_estudis_23_pivot["Sense estudis_edu_Espanya"] + df_estudis_23_pivot["Estudis primaris, certificat d'escolaritat, EGB_edu_Espanya"]
df_estudis_23_pivot["estudis_inferiors_ue"] = df_estudis_23_pivot["Sense estudis_edu_Resta de la Unió Europea"] + df_estudis_23_pivot["Estudis primaris, certificat d'escolaritat, EGB_edu_Resta de la Unió Europea"]
df_estudis_23_pivot["estudis_inferiors_mon"] = df_estudis_23_pivot["Sense estudis_edu_Resta del món"] + df_estudis_23_pivot["Estudis primaris, certificat d'escolaritat, EGB_edu_Resta del món"]

# Eliminem columnes redundants
df_estudis_23_clean = df_estudis_23_pivot.drop(columns= [col for col in df_estudis_23_pivot.columns if not col.startswith("estudis_superiors") and not col.startswith("estudis_inferiors") and col != "Codi_Barri"])
df_estudis_23_clean.head()

,Codi_Barri,estudis_superiors_espanya,estudis_superiors_ue,estudis_superiors_mon,estudis_inferiors_espanya,estudis_inferiors_ue,estudis_inferiors_mon
0,1,3113,2094,5839,3064,171,6173
1,2,1722,1483,3601,992,141,3719
2,3,1010,1192,2155,1835,59,618
3,4,2569,2104,4282,1615,102,1074
4,5,6988,1387,4628,2123,87,1211


In [62]:
# Netejem per preparar per als joins posteriors
# Canvi de noms
df_estudis_15_pivot = df_estudis_15_pivot.rename(columns = 
                                                 {"Estudis universitaris, CFGS grau superior_edu_Espanya": "estudis_superiors_espanya", 
                                                  "Estudis universitaris, CFGS grau superior_edu_Resta de la Unió Europea": "estudis_superiors_ue", 
                                                  "Estudis universitaris, CFGS grau superior_edu_Resta del món": "estudis_superiors_mon"
                                                  })

# Càlcul de columnes
df_estudis_15_pivot["estudis_inferiors_espanya"] = df_estudis_15_pivot["Sense estudis_edu_Espanya"] + df_estudis_15_pivot["Estudis primaris, certificat d'escolaritat, EGB_edu_Espanya"]
df_estudis_15_pivot["estudis_inferiors_ue"] = df_estudis_15_pivot["Sense estudis_edu_Resta de la Unió Europea"] + df_estudis_15_pivot["Estudis primaris, certificat d'escolaritat, EGB_edu_Resta de la Unió Europea"]
df_estudis_15_pivot["estudis_inferiors_mon"] = df_estudis_15_pivot["Sense estudis_edu_Resta del món"] + df_estudis_15_pivot["Estudis primaris, certificat d'escolaritat, EGB_edu_Resta del món"]

# Eliminem columnes redundants
df_estudis_15_clean = df_estudis_15_pivot.drop(columns= [col for col in df_estudis_15_pivot.columns if not col.startswith("estudis_superiors") and not col.startswith("estudis_inferiors") and col != "Codi_Barri"])
df_estudis_15_clean.head()

,Codi_Barri,estudis_superiors_espanya,estudis_superiors_ue,estudis_superiors_mon,estudis_inferiors_espanya,estudis_inferiors_ue,estudis_inferiors_mon
0,1,3192,1977,3552,5422,311,9513
1,2,1962,1485,1728,1695,152,1154
2,3,976,1080,1173,3160,93,831
3,4,2717,2172,2377,2751,103,1709
4,5,6784,1060,2061,3407,111,1062


Aquests dos datasets seran afegits al dataset final de dades poblacionals, i doncs s'extrauran els percentatges.

## Grups per edat i lloc de naixement (Esp, Resta Unió Europea i Resta del món)
Aquestes dades ens permetran estudiar les edats per barris segons el seu origen. Extraurem dues variables:
- pct_joves_esp
- pct_jobes_ue

Amb la segona variable podrem analitzar i estudiar gentrificació causada per Expats. I quin efecte està tenint sobre la població jove espanyola. En aquest cas, població jove es considerarà desde edats de 20 a 40 anys (No inclòs):
- 4 -->	20-24 anys
- 5	--> 25-29 anys
- 6 -->	30-34 anys
- 7 -->	35-39 anys

In [63]:
dimensions[dimensions["Desc_Dimensio"] == "EDAT_Q"]

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN
104,3,EDAT_Q,0,<5 anys,<5 años,<5 years
105,3,EDAT_Q,1,5-9 anys,5-9 años,5-9 years
106,3,EDAT_Q,2,10-14 anys,10-14 años,10-14 years
107,3,EDAT_Q,3,15-19 anys,15-19 años,15-19 years
108,3,EDAT_Q,4,20-24 anys,20-24 años,20-24 years
109,3,EDAT_Q,5,25-29 anys,25-29 años,25-29 years
110,3,EDAT_Q,6,30-34 anys,30-34 años,30-34 years
111,3,EDAT_Q,7,35-39 anys,35-39 años,35-39 years
112,3,EDAT_Q,8,40-44 anys,40-44 años,40-44 years
113,3,EDAT_Q,9,45-49 anys,45-49 años,45-49 years


In [64]:
df_edat_raw_23 = pd.read_csv("../data/raw/edat/2023_pad_mdb_lloc-naix_edat-q_sexe.csv")
df_edat_raw_23.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,LLOC_NAIX,EDAT_Q,SEXE
0,2023-01-01,1,Ciutat Vella,1,el Raval,801,1,0,1
1,2023-01-01,1,Ciutat Vella,1,el Raval,868,1,0,2
2,2023-01-01,1,Ciutat Vella,1,el Raval,601,1,1,1
3,2023-01-01,1,Ciutat Vella,1,el Raval,734,1,1,2
4,2023-01-01,1,Ciutat Vella,1,el Raval,577,1,2,1


In [65]:
df_edat_raw_23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14583 entries, 0 to 14582
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  14583 non-null  object
 1   Codi_Districte   14583 non-null  int64 
 2   Nom_Districte    14583 non-null  object
 3   Codi_Barri       14583 non-null  int64 
 4   Nom_Barri        14583 non-null  object
 5   Valor            14583 non-null  object
 6   LLOC_NAIX        14583 non-null  int64 
 7   EDAT_Q           14583 non-null  int64 
 8   SEXE             14583 non-null  int64 
dtypes: int64(5), object(4)
memory usage: 1.0+ MB


**Observacions**
- No hi ha nuls.
- Variable valor format object, indicant presència de ".."

In [67]:
df_edat_raw_15 = pd.read_csv("../data/raw/edat/2015_pad_mdb_lloc-naix_edat-q_sexe.csv")
df_edat_raw_15.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,LLOC_NAIX,EDAT_Q,SEXE
0,2015-01-01,1,Ciutat Vella,1,el Raval,952,1,0,1
1,2015-01-01,1,Ciutat Vella,1,el Raval,980,1,0,2
2,2015-01-01,1,Ciutat Vella,1,el Raval,681,1,1,1
3,2015-01-01,1,Ciutat Vella,1,el Raval,753,1,1,2
4,2015-01-01,1,Ciutat Vella,1,el Raval,454,1,2,1


In [69]:
df_edat_raw_15.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14236 entries, 0 to 14235
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data_Referencia  14236 non-null  object
 1   Codi_Districte   14236 non-null  int64 
 2   Nom_Districte    14236 non-null  object
 3   Codi_Barri       14236 non-null  int64 
 4   Nom_Barri        14236 non-null  object
 5   Valor            14236 non-null  object
 6   LLOC_NAIX        14236 non-null  int64 
 7   EDAT_Q           14236 non-null  int64 
 8   SEXE             14236 non-null  int64 
dtypes: int64(5), object(4)
memory usage: 1001.1+ KB


In [70]:
# Convertim valors ".." a NaN 
df_edat_raw_23["Valor"] = df_edat_raw_23["Valor"].replace("..", 4)
df_edat_raw_15["Valor"] = df_edat_raw_15["Valor"].replace("..", 4)

# Convertir la columna "Valor" a int32
df_edat_raw_23["Valor"] = df_edat_raw_23["Valor"].astype(str).astype("Int32")
df_edat_raw_15["Valor"] = df_edat_raw_15["Valor"].astype(str).astype("Int32")

In [71]:
# Agrupem per barri, grup d'edat i lloc de naixement. Previament filtrarem per als grups d'edat que ens interessen definits anteriorment
scope_edat = range(4,8,1)
df_edat_raw_23_filt = df_edat_raw_23[df_edat_raw_23["EDAT_Q"].isin(scope_edat)]
df_edat_raw_15_filt = df_edat_raw_15[df_edat_raw_15["EDAT_Q"].isin(scope_edat)]

In [72]:
df_edat_raw_23_filt.head()

,Data_Referencia,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Valor,LLOC_NAIX,EDAT_Q,SEXE
8,2023-01-01,1,Ciutat Vella,1,el Raval,313,1,4,1
9,2023-01-01,1,Ciutat Vella,1,el Raval,333,1,4,2
10,2023-01-01,1,Ciutat Vella,1,el Raval,229,1,5,1
11,2023-01-01,1,Ciutat Vella,1,el Raval,243,1,5,2
12,2023-01-01,1,Ciutat Vella,1,el Raval,208,1,6,1


Ja que ara tenim el dataset filtrat per el scope de persones joves, l'agrupament el farem per barri i lloc de naixement.

In [73]:
df_edat_raw_23_agg = df_edat_raw_23_filt.groupby(["Codi_Barri", "LLOC_NAIX"]).agg({"Valor": "sum"}).reset_index()
df_edat_raw_15_agg = df_edat_raw_15_filt.groupby(["Codi_Barri", "LLOC_NAIX"]).agg({"Valor": "sum"}).reset_index()

In [74]:
# Creuem amb la dimensió de lloc de naixement per obtenir les descripcions
df_edat_23_merged = df_edat_raw_23_agg.merge(dim_lloc_naix[["Codi_Valor","Desc_Valor_CA", "custom_mapping_regio"]], left_on="LLOC_NAIX", right_on="Codi_Valor", how="left")
df_edat_15_merged = df_edat_raw_15_agg.merge(dim_lloc_naix[["Codi_Valor","Desc_Valor_CA", "custom_mapping_regio"]], left_on="LLOC_NAIX", right_on="Codi_Valor", how="left")

df_edat_23_merged.head()

,Codi_Barri,LLOC_NAIX,Valor,Codi_Valor,Desc_Valor_CA,custom_mapping_regio
0,1,1,1951,1,Barcelona ciutat,Espanya
1,1,2,464,2,Resta de Catalunya,Espanya
2,1,3,838,3,Resta d’Espanya,Espanya
3,1,4,2362,4,Resta de la Unió Europea,Resta de la Unió Europea
4,1,5,11771,5,Resta del món,Resta del món


In [75]:
df_edat_23_pivot = df_edat_23_merged.pivot_table(index=["Codi_Barri"], columns="custom_mapping_regio", values="Valor", aggfunc= "sum", fill_value=0).reset_index()
df_edat_15_pivot = df_edat_15_merged.pivot_table(index=["Codi_Barri"], columns="custom_mapping_regio", values="Valor", aggfunc= "sum", fill_value=0).reset_index()
df_edat_23_pivot.head()

custom_mapping_regio,Codi_Barri,Espanya,No Consta,Resta de la Unió Europea,Resta del món
0,1,3253,29,2362,11771
1,2,1435,16,1667,9196
2,3,1465,4,1369,2992
3,4,2016,16,2189,5049
4,5,3851,20,1270,6043


In [76]:
df_edat_23_pivot.columns.name = None
df_edat_23_pivot = df_edat_23_pivot.reset_index(drop=True)

df_edat_15_pivot.columns.name = None
df_edat_15_pivot = df_edat_15_pivot.reset_index(drop=True)

In [77]:
df_edat_23_pivot.columns = [col if col == 'Codi_Barri' else f"joves_{col}" for col in df_edat_23_pivot.columns]
df_edat_15_pivot.columns = [col if col == 'Codi_Barri' else f"joves_{col}" for col in df_edat_15_pivot.columns]

df_edat_23_pivot.head()

,Codi_Barri,joves_Espanya,joves_No Consta,joves_Resta de la Unió Europea,joves_Resta del món
0,1,3253,29,2362,11771
1,2,1435,16,1667,9196
2,3,1465,4,1369,2992
3,4,2016,16,2189,5049
4,5,3851,20,1270,6043


In [78]:
# Agreguem columna amb total de joves
cols_joves = [col for col in df_edat_23_pivot.columns if col.startswith("joves_")]
df_edat_23_pivot["joves_total"] = df_edat_23_pivot[cols_joves].sum(axis=1)
df_edat_15_pivot["joves_total"] = df_edat_15_pivot[cols_joves].sum(axis=1)

## Agregacions
En aquest apartat final de les dades demogràfiques, agregarem els conjunts en un sol

### 2023

In [79]:
df_demografia = dim_barris.copy()
df_demografia.rename(columns={"codi_barri": "Codi_Barri"}, inplace=True)

df_demografia_23 =  df_demografia[["Codi_Barri"]].merge(df_poblacio_total_23, on="Codi_Barri", how="left")\
                                        .merge(df_pob_23_pivot, on="Codi_Barri", how="left")\
                                        .merge(df_nacionalitats_23_pivot, on="Codi_Barri", how="left")\
                                        .merge(df_estudis_23_clean, on="Codi_Barri", how="left")\
                                        .merge(df_edat_23_pivot, on="Codi_Barri", how="left")

df_demografia_23.head()

,Codi_Barri,Poblacio_Total,Espanya,No consta_x,Resta de la Unió Europea,Resta del món,Amèrica central,Amèrica del nord,Amèrica del sud,Austràlia i Nova Zelanda,...,estudis_superiors_ue,estudis_superiors_mon,estudis_inferiors_espanya,estudis_inferiors_ue,estudis_inferiors_mon,joves_Espanya,joves_No Consta,joves_Resta de la Unió Europea,joves_Resta del món,joves_total
0,1,45671,22396,56,4563,18693,470,297,2888,16,...,2094,5839,3064,171,6173,3253,29,2362,11771,17415
1,2,24460,8385,32,3281,12784,453,306,2203,32,...,1483,3601,992,141,3719,1435,16,1667,9196,12314
2,3,14274,8264,4,2619,3390,184,129,1248,21,...,1192,2155,1835,59,618,1465,4,1369,2992,5830
3,4,22041,11764,16,4284,5989,313,390,1736,53,...,2104,4282,1615,102,1074,2016,16,2189,5049,9270
4,5,34403,23354,16,3445,7600,608,151,3088,14,...,1387,4628,2123,87,1211,3851,20,1270,6043,11184


In [80]:
# Netejem nom columnes
df_demografia_23_clean =neteja_noms_columnes(df_demografia_23)
df_demografia_23_clean.head()

,codi_barri,poblacio_total,espanya,no_consta_x,resta_de_la_uni_europea,resta_del_mn,amrica_central,amrica_del_nord,amrica_del_sud,austrlia_i_nova_zelanda,...,estudis_superiors_ue,estudis_superiors_mon,estudis_inferiors_espanya,estudis_inferiors_ue,estudis_inferiors_mon,joves_espanya,joves_no_consta,joves_resta_de_la_uni_europea,joves_resta_del_mn,joves_total
0,1,45671,22396,56,4563,18693,470,297,2888,16,...,2094,5839,3064,171,6173,3253,29,2362,11771,17415
1,2,24460,8385,32,3281,12784,453,306,2203,32,...,1483,3601,992,141,3719,1435,16,1667,9196,12314
2,3,14274,8264,4,2619,3390,184,129,1248,21,...,1192,2155,1835,59,618,1465,4,1369,2992,5830
3,4,22041,11764,16,4284,5989,313,390,1736,53,...,2104,4282,1615,102,1074,2016,16,2189,5049,9270
4,5,34403,23354,16,3445,7600,608,151,3088,14,...,1387,4628,2123,87,1211,3851,20,1270,6043,11184


### 2015

In [81]:
df_demografia_15 =  df_demografia[["Codi_Barri"]].merge(df_poblacio_total_15, on="Codi_Barri", how="left")\
                                        .merge(df_pob_15_pivot, on="Codi_Barri", how="left")\
                                        .merge(df_nacionalitats_15_pivot, on="Codi_Barri", how="left")\
                                        .merge(df_estudis_15_clean, on="Codi_Barri", how="left")\
                                        .merge(df_edat_15_pivot, on="Codi_Barri", how="left")

df_demografia_15.head()

,Codi_Barri,Poblacio_Total,Espanya,No consta_x,Resta de la Unió Europea,Resta del món,Amèrica central,Amèrica del nord,Amèrica del sud,Austràlia i Nova Zelanda,...,estudis_superiors_ue,estudis_superiors_mon,estudis_inferiors_espanya,estudis_inferiors_ue,estudis_inferiors_mon,joves_Espanya,joves_No Consta,joves_Resta de la Unió Europea,joves_Resta del món,joves_total
0,1,47150,24498,44,4347,18293,292,177,1862,12,...,1977,3552,5422,311,9513,4240,24,2787,11528,18579
1,2,15514,9007,20,3079,3422,182,164,772,24,...,1485,1728,1695,152,1154,1845,12,1873,2912,6642
2,3,15037,10331,4,2178,2527,87,95,734,17,...,1080,1173,3160,93,831,2042,4,1422,2144,5612
3,4,22468,13644,44,4146,4666,221,227,1121,41,...,2172,2377,2751,103,1709,2789,20,2461,3803,9073
4,5,31548,25417,36,2340,3778,261,72,1247,8,...,1060,2061,3407,111,1062,4997,20,1163,3165,9345


In [82]:
# Netejem nom columnes
df_demografia_15_clean =neteja_noms_columnes(df_demografia_15)
df_demografia_15_clean.head()

,codi_barri,poblacio_total,espanya,no_consta_x,resta_de_la_uni_europea,resta_del_mn,amrica_central,amrica_del_nord,amrica_del_sud,austrlia_i_nova_zelanda,...,estudis_superiors_ue,estudis_superiors_mon,estudis_inferiors_espanya,estudis_inferiors_ue,estudis_inferiors_mon,joves_espanya,joves_no_consta,joves_resta_de_la_uni_europea,joves_resta_del_mn,joves_total
0,1,47150,24498,44,4347,18293,292,177,1862,12,...,1977,3552,5422,311,9513,4240,24,2787,11528,18579
1,2,15514,9007,20,3079,3422,182,164,772,24,...,1485,1728,1695,152,1154,1845,12,1873,2912,6642
2,3,15037,10331,4,2178,2527,87,95,734,17,...,1080,1173,3160,93,831,2042,4,1422,2144,5612
3,4,22468,13644,44,4146,4666,221,227,1121,41,...,2172,2377,2751,103,1709,2789,20,2461,3803,9073
4,5,31548,25417,36,2340,3778,261,72,1247,8,...,1060,2061,3407,111,1062,4997,20,1163,3165,9345


In [92]:
df_demografia_23_clean.to_csv("../data/processed/df_demografia_23.csv", index=False)

In [ ]:
df_demografia_15_clean.to_csv("../data/processed/df_demografia_15.csv", index=False)